# Stage 3 — Predictive Data Analysis (PDA)
## NYC Yellow Taxi 2025 — Generous Tip Classification

**Task**: Binary classification — predict whether a passenger tips more than 20% of the fare.

**Models**:
1. Random Forest Classifier
2. Gradient-Boosted Trees (GBT) Classifier
3. Logistic Regression

**Pipeline**:
- Feature engineering from raw parquet files
- VectorAssembler + StandardScaler
- CrossValidator with ParamGridBuilder (3 hyperparameters × 3 values each)
- Evaluation: Area Under ROC, Area Under PR
- Best model saved to `models/`
- Results saved to `output/`

## 0. Setup & Spark Session

In [ ]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, Imputer
from pyspark.ml.classification import (
    RandomForestClassifier,
    GBTClassifier,
    LogisticRegression,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Spark ──────────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName('NYC_Taxi_Stage3_ML')
    # When running on YARN cluster, remove master() — submit with spark-submit --master yarn
    # .master('yarn')
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('Data dir:', DATA_DIR)

## 1. Load Data

In [ ]:
# Load all 12 months of NYC Yellow Taxi 2025
raw_df = spark.read.parquet(os.path.join(DATA_DIR, 'yellow_tripdata_2025-*.parquet'))

print(f'Total records: {raw_df.count():,}')
print(f'Columns ({len(raw_df.columns)}): {raw_df.columns}')
raw_df.printSchema()

## 2. Feature Engineering

In [ ]:
def engineer_features(df):
    """
    Build ML-ready features from raw taxi trip data.
    
    Label: generous_tip = 1 if tip_amount / fare_amount > 0.20 else 0
    Only credit-card trips (payment_type == 1) have reliable tip data.
    """
    df = (
        df
        # Keep only credit-card payments (tip is auto-recorded)
        .filter(F.col('payment_type') == 1)
        # Drop rows with non-positive fare or negative trip distance
        .filter(F.col('fare_amount') > 0)
        .filter(F.col('trip_distance') > 0)
        .filter(F.col('tip_amount') >= 0)
        # Drop obvious outliers
        .filter(F.col('fare_amount') < 500)
        .filter(F.col('trip_distance') < 200)
        .filter(F.col('passenger_count').between(1, 6))
        # Temporal features
        .withColumn('trip_duration_min',
            (F.unix_timestamp('tpep_dropoff_datetime') -
             F.unix_timestamp('tpep_pickup_datetime')) / 60.0)
        .filter(F.col('trip_duration_min').between(1, 180))
        .withColumn('pickup_hour',      F.hour('tpep_pickup_datetime').cast(DoubleType()))
        .withColumn('pickup_dayofweek', F.dayofweek('tpep_pickup_datetime').cast(DoubleType()))
        .withColumn('pickup_month',     F.month('tpep_pickup_datetime').cast(DoubleType()))
        .withColumn('is_weekend',
            F.when(F.dayofweek('tpep_pickup_datetime').isin([1, 7]), 1.0).otherwise(0.0))
        # Speed proxy
        .withColumn('avg_speed_mph',
            F.col('trip_distance') / (F.col('trip_duration_min') / 60.0))
        # Label
        .withColumn('tip_ratio', F.col('tip_amount') / F.col('fare_amount'))
        .withColumn('label',
            F.when(F.col('tip_ratio') > 0.20, 1.0).otherwise(0.0))
    )
    return df


engineered_df = engineer_features(raw_df)

FEATURE_COLS = [
    'VendorID',
    'passenger_count',
    'trip_distance',
    'RatecodeID',
    'PULocationID',
    'DOLocationID',
    'fare_amount',
    'extra',
    'mta_tax',
    'tolls_amount',
    'congestion_surcharge',
    'airport_fee',
    'trip_duration_min',
    'pickup_hour',
    'pickup_dayofweek',
    'pickup_month',
    'is_weekend',
    'avg_speed_mph',
]

ml_df = engineered_df.select(FEATURE_COLS + ['label']).cache()

total = ml_df.count()
pos   = ml_df.filter(F.col('label') == 1).count()
print(f'ML dataset size : {total:,}')
print(f'Positive (tip>20%): {pos:,} ({100*pos/total:.1f}%)')
print(f'Negative          : {total-pos:,} ({100*(total-pos)/total:.1f}%)')

## 3. Train / Test Split (70 / 30)

In [ ]:
train_df, test_df = ml_df.randomSplit([0.70, 0.30], seed=42)
train_df = train_df.cache()
test_df  = test_df.cache()

print(f'Train size: {train_df.count():,}')
print(f'Test  size: {test_df.count():,}')

## 4. Shared Pre-processing Pipeline Stages

In [ ]:
# Impute nulls with median strategy
imputer = Imputer(
    inputCols=FEATURE_COLS,
    outputCols=[c + '_imp' for c in FEATURE_COLS],
    strategy='median',
)

imputed_cols = [c + '_imp' for c in FEATURE_COLS]

assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol='features_raw',
    handleInvalid='skip',
)

scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True,
)

preprocessing_stages = [imputer, assembler, scaler]
print('Preprocessing stages ready.')

## 5. Evaluators

In [ ]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC',
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderPR',
)

results = {}  # will store {model_name: {auroc, aupr}}

---
## 6. Model 1 — Random Forest Classifier

**Hyperparameters tuned** (3 values each = 27 combinations):
- `numTrees`: [50, 100, 200]
- `maxDepth`: [5, 10, 15]
- `minInstancesPerNode`: [1, 5, 10]

In [ ]:
rf = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    seed=42,
)

rf_pipeline = Pipeline(stages=preprocessing_stages + [rf])

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees,             [50, 100, 200])       # algorithm hyperparameter
    .addGrid(rf.maxDepth,             [5, 10, 15])          # model hyperparameter
    .addGrid(rf.minInstancesPerNode,  [1, 5, 10])           # model hyperparameter
    .build()
)

print(f'RF grid size: {len(rf_param_grid)} combinations')

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator_roc,
    numFolds=3,          # k=3 (2 < k < 5)
    seed=42,
    parallelism=4,
)

print('Training Random Forest with 3-fold CV...')
rf_cv_model = rf_cv.fit(train_df)
print('Done.')

In [ ]:
rf_best = rf_cv_model.bestModel
rf_preds = rf_best.transform(test_df)

rf_auroc = evaluator_roc.evaluate(rf_preds)
rf_aupr  = evaluator_pr.evaluate(rf_preds)

# Best hyperparameters
rf_stage = rf_best.stages[-1]   # RandomForestClassificationModel
rf_best_params = {
    'numTrees':            rf_stage.getNumTrees,
    'maxDepth':            rf_stage.getOrDefault(rf_stage.maxDepth),
    'minInstancesPerNode': rf_stage.getOrDefault(rf_stage.minInstancesPerNode),
}

results['RandomForest'] = {'AUROC': rf_auroc, 'AUPR': rf_aupr, 'best_params': rf_best_params}

print('=== Random Forest Results ===')
print(f'  Area Under ROC : {rf_auroc:.4f}')
print(f'  Area Under PR  : {rf_aupr:.4f}')
print(f'  Best params    : {rf_best_params}')

# Feature importances
importances = rf_stage.featureImportances.toArray()
fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)
print('\nTop 10 features:')
print(fi_df.head(10).to_string(index=False))

In [ ]:
# Save best RF model
rf_model_path = os.path.join(MODELS_DIR, 'random_forest_model')
rf_best.write().overwrite().save(rf_model_path)
print(f'RF model saved to {rf_model_path}')

# Feature importance plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'][:15][::-1], fi_df['importance'][:15][::-1], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Top 15 Feature Importances')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'rf_feature_importance.png'), dpi=150)
plt.close(fig)
print('Feature importance plot saved.')

---
## 7. Model 2 — Gradient-Boosted Trees (GBT) Classifier

**Hyperparameters tuned** (3 values each = 27 combinations):
- `maxIter`: [10, 20, 30]
- `maxDepth`: [3, 5, 7]
- `stepSize` (learning rate): [0.05, 0.10, 0.20]

In [ ]:
gbt = GBTClassifier(
    labelCol='label',
    featuresCol='features',
    seed=42,
)

gbt_pipeline = Pipeline(stages=preprocessing_stages + [gbt])

gbt_param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter,  [10, 20, 30])          # algorithm hyperparameter
    .addGrid(gbt.maxDepth, [3, 5, 7])              # model hyperparameter
    .addGrid(gbt.stepSize, [0.05, 0.10, 0.20])     # model hyperparameter
    .build()
)

print(f'GBT grid size: {len(gbt_param_grid)} combinations')

gbt_cv = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_param_grid,
    evaluator=evaluator_roc,
    numFolds=3,
    seed=42,
    parallelism=4,
)

print('Training GBT with 3-fold CV...')
gbt_cv_model = gbt_cv.fit(train_df)
print('Done.')

In [ ]:
gbt_best = gbt_cv_model.bestModel
gbt_preds = gbt_best.transform(test_df)

gbt_auroc = evaluator_roc.evaluate(gbt_preds)
gbt_aupr  = evaluator_pr.evaluate(gbt_preds)

gbt_stage = gbt_best.stages[-1]
gbt_best_params = {
    'maxIter':  gbt_stage.getOrDefault(gbt_stage.maxIter),
    'maxDepth': gbt_stage.getOrDefault(gbt_stage.maxDepth),
    'stepSize': gbt_stage.getOrDefault(gbt_stage.stepSize),
}

results['GBT'] = {'AUROC': gbt_auroc, 'AUPR': gbt_aupr, 'best_params': gbt_best_params}

print('=== GBT Results ===')
print(f'  Area Under ROC : {gbt_auroc:.4f}')
print(f'  Area Under PR  : {gbt_aupr:.4f}')
print(f'  Best params    : {gbt_best_params}')

In [ ]:
gbt_model_path = os.path.join(MODELS_DIR, 'gbt_model')
gbt_best.write().overwrite().save(gbt_model_path)
print(f'GBT model saved to {gbt_model_path}')

# GBT feature importances
gbt_importances = gbt_stage.featureImportances.toArray()
gbt_fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': gbt_importances})
gbt_fi_df = gbt_fi_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(gbt_fi_df['feature'][:15][::-1], gbt_fi_df['importance'][:15][::-1], color='darkorange')
ax.set_xlabel('Importance')
ax.set_title('GBT — Top 15 Feature Importances')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'gbt_feature_importance.png'), dpi=150)
plt.close(fig)
print('GBT feature importance plot saved.')

---
## 8. Model 3 — Logistic Regression

**Hyperparameters tuned** (3 values each = 27 combinations):
- `regParam` (regularization λ): [0.001, 0.01, 0.1]
- `elasticNetParam` (L1 vs L2 mix): [0.0, 0.5, 1.0]
- `maxIter`: [50, 100, 200]

In [ ]:
lr = LogisticRegression(
    labelCol='label',
    featuresCol='features',
)

lr_pipeline = Pipeline(stages=preprocessing_stages + [lr])

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,        [0.001, 0.01, 0.1])   # algorithm hyperparameter
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])      # model hyperparameter
    .addGrid(lr.maxIter,         [50, 100, 200])        # model hyperparameter
    .build()
)

print(f'LR grid size: {len(lr_param_grid)} combinations')

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator_roc,
    numFolds=3,
    seed=42,
    parallelism=4,
)

print('Training Logistic Regression with 3-fold CV...')
lr_cv_model = lr_cv.fit(train_df)
print('Done.')

In [ ]:
lr_best = lr_cv_model.bestModel
lr_preds = lr_best.transform(test_df)

lr_auroc = evaluator_roc.evaluate(lr_preds)
lr_aupr  = evaluator_pr.evaluate(lr_preds)

lr_stage = lr_best.stages[-1]
lr_best_params = {
    'regParam':        lr_stage.getOrDefault(lr_stage.regParam),
    'elasticNetParam': lr_stage.getOrDefault(lr_stage.elasticNetParam),
    'maxIter':         lr_stage.getOrDefault(lr_stage.maxIter),
}

results['LogisticRegression'] = {'AUROC': lr_auroc, 'AUPR': lr_aupr, 'best_params': lr_best_params}

print('=== Logistic Regression Results ===')
print(f'  Area Under ROC : {lr_auroc:.4f}')
print(f'  Area Under PR  : {lr_aupr:.4f}')
print(f'  Best params    : {lr_best_params}')

In [ ]:
lr_model_path = os.path.join(MODELS_DIR, 'logistic_regression_model')
lr_best.write().overwrite().save(lr_model_path)
print(f'LR model saved to {lr_model_path}')

---
## 9. Model Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {'Model': name, 'AUROC': v['AUROC'], 'AUPR': v['AUPR']}
    for name, v in results.items()
]).sort_values('AUROC', ascending=False)

print('\n=== Final Model Comparison ===')
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.iloc[0]['Model']
print(f'\nBest model by AUROC: {best_model_name}')

In [ ]:
# Bar chart: AUROC & AUPR comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['steelblue', 'darkorange', 'seagreen']

axes[0].bar(comparison_df['Model'], comparison_df['AUROC'], color=colors)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('Area Under ROC')
axes[0].set_ylabel('AUROC')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['AUROC']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

axes[1].bar(comparison_df['Model'], comparison_df['AUPR'], color=colors)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('Area Under Precision-Recall')
axes[1].set_ylabel('AUPR')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['AUPR']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('NYC Taxi Tip Classification — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150)
plt.close(fig)
print('Model comparison chart saved.')

---
## 10. Predict a Specific Sample

Demonstrate prediction on a single trip record.

In [ ]:
# Pick the best model pipeline
best_pipeline = {
    'RandomForest':      rf_best,
    'GBT':               gbt_best,
    'LogisticRegression': lr_best,
}[best_model_name]

# A sample trip: 2.5 miles, 1 passenger, 10 min, evening rush hour on a weekday
sample_data = [{
    'VendorID':            2.0,
    'passenger_count':     1.0,
    'trip_distance':       2.5,
    'RatecodeID':          1.0,
    'PULocationID':        161.0,   # Midtown Center
    'DOLocationID':        236.0,   # Upper East Side South
    'fare_amount':         12.5,
    'extra':               1.0,
    'mta_tax':             0.5,
    'tolls_amount':        0.0,
    'congestion_surcharge': 2.5,
    'airport_fee':         0.0,
    'trip_duration_min':   10.0,
    'pickup_hour':         18.0,
    'pickup_dayofweek':    3.0,     # Wednesday
    'pickup_month':        6.0,
    'is_weekend':          0.0,
    'avg_speed_mph':       15.0,
    'label':               -1.0,   # unknown
}]

sample_df = spark.createDataFrame(sample_data)
prediction = best_pipeline.transform(sample_df)

pred_label = prediction.select('prediction', 'probability').collect()[0]
prob_generous = float(pred_label['probability'][1])

print(f'Sample trip prediction ({best_model_name}):')
print(f'  Predicted label : {int(pred_label["prediction"])} ({"Generous tip (>20%)" if pred_label["prediction"]==1 else "Not generous tip"})')
print(f'  P(generous tip) : {prob_generous:.4f}')

## 11. Save Results to output/

In [ ]:
# Save comparison CSV
comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'stage3_model_comparison.csv'), index=False)

# Save full results JSON
results_serializable = {
    name: {
        'AUROC': float(v['AUROC']),
        'AUPR':  float(v['AUPR']),
        'best_params': {k: (int(p) if isinstance(p, (int,)) else float(p) if isinstance(p, float) else str(p))
                        for k, p in v['best_params'].items()},
    }
    for name, v in results.items()
}
results_serializable['best_model'] = best_model_name
results_serializable['sample_prediction'] = {
    'label': int(pred_label['prediction']),
    'prob_generous_tip': prob_generous,
}

with open(os.path.join(OUTPUT_DIR, 'stage3_results.json'), 'w') as f:
    json.dump(results_serializable, f, indent=2)

print('Results saved:')
print(f'  {OUTPUT_DIR}/stage3_model_comparison.csv')
print(f'  {OUTPUT_DIR}/stage3_results.json')
print(f'  {OUTPUT_DIR}/rf_feature_importance.png')
print(f'  {OUTPUT_DIR}/gbt_feature_importance.png')
print(f'  {OUTPUT_DIR}/model_comparison.png')

In [ ]:
spark.stop()
print('Spark session stopped.')